# Object Recognition with Open Images v7 and Webcam

In this notebook we prepare a dataset from **Open Images v7** using **FiftyOne**, crop the annotated objects with **OpenCV**, then adapt a **ResNet50** model with transfer learning and fine-tuning in **TensorFlow/Keras** to recognize everyday objects on a live webcam feed.

**Steps:**
1. Import libraries
2. Choose target classes
3. Download the train and validation subsets
4. Crop objects and check class balance *(next)*
5. Transfer learning with a frozen ResNet50 *(next)*
6. Fine-tuning the top layers *(next)*
7. Evaluate and run the live webcam demo *(next)*

In [2]:
import fiftyone as fo
import fiftyone.zoo as foz

In [3]:
print(fo.list_datasets())

You are running the oldest supported major version of MongoDB. Please refer to https://deprecation.voxel51.com for deprecation notices. You can suppress this exception by setting your `database_validation` config parameter to `False`. See https://docs.voxel51.com/user_guide/config.html#configuring-a-mongodb-connection for more information
['train_dataset']


## 1. Target classes

We pick **30 everyday classes** that are likely to appear in front of a webcam:

- **Electronics:** mobile phone, laptop, keyboard, mouse, headphones, monitor, remote control, camera
- **Office items:** book, pen, scissors, calculator
- **Kitchen and tableware:** mug, bottle, coffee cup, plate, bowl, spoon, fork, knife
- **Personal items:** glasses, watch, backpack, clock
- **Fruit:** banana, apple, orange
- **People:** person, human face, human hand

Class names must match the Open Images names exactly (capitalized as in the official list). The next cell prints the number of classes as a sanity check.

In [1]:
labels = [
    "Mobile phone", "Laptop", "Computer keyboard", "Computer mouse",
    "Headphones", "Computer monitor", "Remote control", "Camera",
    "Book", "Pen", "Scissors", "Calculator",
    "Mug", "Bottle", "Coffee cup", "Plate", "Bowl", "Spoon", "Fork", "Knife",
    "Glasses", "Watch", "Backpack", "Clock",
    "Banana", "Apple", "Orange",
    "Person", "Human face", "Human hand",
]
print(len(labels))

30


## 2. Download the dataset

We download the **train** split of Open Images v7 with bounding-box labels (`detections`), **one class at a time**, so every class gets a similar number of images instead of letting frequent classes (Person, Clothing, ...) dominate.

- `classes=[label]`: only images containing the current class (one class per iteration)
- `max_samples=1000`: images per class (about 30,000 images in total for 96 classes)
- `shuffle=True`, `seed=i`: random but reproducible selection
- `dataset_name`: a temporary dataset per class, merged into `train_dataset` and then deleted

> The first run downloads large metadata files (about 4.8 GB), so it can take several minutes. Later runs use the local cache in `~/fiftyone/open-images-v7`. If the download is interrupted, run the cell again: already downloaded images are not downloaded twice.

In [ ]:
if fo.dataset_exists("train_dataset"):
    fo.delete_dataset("train_dataset")
train_ds = fo.Dataset("train_dataset", persistent=True)

for i, label in enumerate(labels):
    tmp = f"tmp_train_{i}"
    if fo.dataset_exists(tmp):
        fo.delete_dataset(tmp)

    ds = foz.load_zoo_dataset(
        "open-images-v7",
        split="train",
        classes=[label],
        label_types=["detections"],
        max_samples=1000,
        shuffle=True,
        seed=i,
        dataset_name=tmp,
    )
    train_ds.add_samples(ds)
    ds.delete()
    print(i, label, len(train_ds))

 100% |█████████████████| 1000/1000 [4.3m elapsed, 0s remaining, 4.5 files/s]      
Dataset info written to '/home/amirm/fiftyone/open-images-v7/info.json'
Loading 'open-images-v7' split 'train'
 100% |███████████████| 1000/1000 [4.3s elapsed, 0s remaining, 247.0 samples/s]      
Dataset 'tmp_train_0' created
 100% |███████████████| 1000/1000 [4.1s elapsed, 0s remaining, 258.0 samples/s]      
0 Mobile phone 1000
Found 5 images, downloading the remaining 995
 100% |███████████████████| 995/995 [4.9m elapsed, 0s remaining, 3.7 files/s]      
Dataset info written to '/home/amirm/fiftyone/open-images-v7/info.json'
Loading 'open-images-v7' split 'train'
 100% |███████████████| 1000/1000 [7.9s elapsed, 0s remaining, 111.0 samples/s]      
Dataset 'tmp_train_1' created
 100% |███████████████| 1000/1000 [8.8s elapsed, 0s remaining, 106.2 samples/s]      
1 Laptop 2000
Found 80 images, downloading the remaining 920
 100% |███████████████████| 920/920 [3.8m elapsed, 0s remaining, 3.3 files/s]  

## 3. Download the validation split

We download the **validation** split of Open Images v7 the same way as the train split: one class at a time, same target classes, same label type (`detections`), so the model can be evaluated on images it has never seen during training.

- `split="validation"`: the validation part of Open Images v7
- `classes=[label]`: only images containing the current class (one class per iteration)
- `max_samples=300`: images per class (about 9,000 images in total, about 30% of the training size)
- `shuffle=True`, `seed=i`: random but reproducible selection
- `dataset_name`: a temporary dataset per class, merged into `validation_dataset` and then deleted

> The train and validation splits are separate in Open Images, so there is no overlap between them. The validation metadata is much smaller than the train metadata, so this step is faster than the previous download. Rare classes may have fewer than 300 images.

In [ ]:
if fo.dataset_exists("validation_dataset"):
    fo.delete_dataset("validation_dataset")
val_ds = fo.Dataset("validation_dataset", persistent=True)

for i, label in enumerate(labels):
    tmp = f"tmp_val_{i}"
    if fo.dataset_exists(tmp):
        fo.delete_dataset(tmp)

    ds = foz.load_zoo_dataset(
        "open-images-v7",
        split="validation",
        classes=[label],
        label_types=["detections"],
        max_samples=300,
        shuffle=True,
        seed=i,
        dataset_name=tmp,
    )
    val_ds.add_samples(ds)
    ds.delete()
    print(i, label, len(val_ds))